# Volume 2, Chapter 8: Field Extensions (Python companion)

Exercises first, solutions at the end. We build exact arithmetic in $\Q(\alpha)=\Q[X]/(m_\alpha)$ from scratch (this is how computer algebra systems compute with algebraic numbers), compute minimal polynomials, search for primitive elements, test which regular polygons pass Wantzel's degree test, and finally play ruler and compass by brute force: how many points can be constructed from $0$ and $1$ in two rounds?

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
from fractions import Fraction
import sympy
from sympy import symbols, sqrt, cbrt, cos, pi, minimal_polynomial, totient, Poly, QQ, nsimplify
import matplotlib.pyplot as plt
X = symbols('X')

## Exercises

**P8.1 (★) Arithmetic in $\Q[X]/(m)$.** Write a small class for elements of $\Q(\alpha)\cong\Q[X]/(m)$, $m$ monic irreducible over $\Q$: elements are lists of $d$ rational coefficients, products are reduced modulo $m$, and inverses come from the extended Euclidean algorithm in $\Q[X]$ (Theorem 8.6). Check Exercise 8.2 ($1/(1+\sqrt[3]2+\sqrt[3]4)=\sqrt[3]2-1$) and Example 8.8 ($\sqrt2=\frac12(\alpha^3-9\alpha)$ for $\alpha=\sqrt2+\sqrt3$, computed inside $\Q[X]/(X^4-10X^2+1)$).

In [ ]:
# Your code here


**P8.2 (★) Minimal polynomials.** With SymPy's `minimal_polynomial`, find the minimal polynomials of $\sqrt2+\sqrt3$, $\sqrt2+\sqrt[3]2$ (Exercise 8.1), $2\cos\frac{2\pi}7$, $2\cos\frac{2\pi}9$ and $\cos\frac{2\pi}{17}$ (Exercise 8.4). Then check numerically that the degree of $2\cos\frac{2\pi}n$ is $\varphi(n)/2$ for $3\le n\le 30$.

In [ ]:
# Your code here


**P8.3 (★★) Which polygons pass Wantzel's test?** Theorem 8.17 says: if the regular $n$-gon is constructible, the degree of $2\cos\frac{2\pi}n$ is a power of $2$. By P8.2 (and a theorem of Gauss, for every $n$), that degree is $\varphi(n)/2$, so $\varphi(n)$ must be a power of $2$. List the $n\le 300$ with $\varphi(n)$ a power of $2$, and compare with the numbers $2^k p_1\cdots p_s$ with distinct Fermat primes $p_i\in\{3,5,17,257,65537\}$. (Chapter 10 proves that these $n$-gons are all constructible: the Gauss–Wantzel theorem.)

In [ ]:
# Your code here


**P8.4 (★★) Primitive elements.** For $\alpha=\sqrt[3]2$ and $\beta=\omega=e^{2i\pi/3}$, compute the degree of $\alpha+c\beta$ over $\Q$ for $c=0,1,\dots,5$, and explain which values of $c$ fail (Exercise 8.8). Then check that $\sqrt2+\sqrt3+\sqrt5$ has degree $8$, and that $\sqrt2+\sqrt3+\sqrt5+\sqrt7$ has degree $16$.

In [ ]:
# Your code here


**P8.5 (★★★) Ruler and compass by brute force.** Start from the points $0$ and $1$. One *round* draws every line through two known points and every circle centred at a known point through another known point, then adds all intersection points. Count the points after one and two rounds (floating point is enough if you deduplicate with a tolerance), and plot them. Then construct the regular pentagon of Figure 8.1 with these primitives and check its vertices.

In [ ]:
# Your code here


---
## Solutions

### P8.1

In [ ]:
def pdivmod(a, b):
    # polynomials as lists of Fractions, lowest degree first
    a = a[:]; q = [Fraction(0)] * max(len(a) - len(b) + 1, 1)
    while len(a) >= len(b) and any(a):
        c = a[-1] / b[-1]; k = len(a) - len(b); q[k] = c
        for i, bi in enumerate(b):
            a[i + k] -= c * bi
        a.pop()
        while a and a[-1] == 0:
            a.pop()
    return q, a

def egcd(a, b):
    # returns g, u, v with u a + v b = g
    r0, r1, u0, u1, v0, v1 = a, b, [Fraction(1)], [], [], [Fraction(1)]
    def sub(x, y):
        n = max(len(x), len(y)); z = [(x[i] if i < len(x) else 0) - (y[i] if i < len(y) else 0) for i in range(n)]
        while z and z[-1] == 0: z.pop()
        return z
    def mul(x, y):
        if not x or not y: return []
        z = [Fraction(0)] * (len(x) + len(y) - 1)
        for i, xi in enumerate(x):
            for j, yj in enumerate(y): z[i + j] += xi * yj
        return z
    while r1:
        q, r = pdivmod(r0, r1)
        r0, r1 = r1, r
        u0, u1 = u1, sub(u0, mul(q, u1))
        v0, v1 = v1, sub(v0, mul(q, v1))
    return r0, u0, v0

class NF:
    # an element of Q[X]/(m), m monic irreducible, given by its coefficients (lowest first)
    def __init__(self, m, coeffs):
        self.m = [Fraction(c) for c in m]; d = len(self.m) - 1
        c = [Fraction(x) for x in coeffs]
        _, r = pdivmod(c, self.m) if len(c) > d else (None, c)
        self.c = (r + [Fraction(0)] * d)[:d]
    def __add__(self, o): o = self._lift(o); return NF(self.m, [a + b for a, b in zip(self.c, o.c)])
    def __sub__(self, o): o = self._lift(o); return NF(self.m, [a - b for a, b in zip(self.c, o.c)])
    def __mul__(self, o):
        o = self._lift(o); z = [Fraction(0)] * (2 * len(self.c))
        for i, a in enumerate(self.c):
            for j, b in enumerate(o.c): z[i + j] += a * b
        return NF(self.m, z)
    __radd__, __rmul__ = __add__, __mul__
    def __pow__(self, k):
        r = NF(self.m, [1])
        for _ in range(k): r = r * self
        return r
    def inv(self):
        a = self.c[:]
        while a and a[-1] == 0: a.pop()
        g, u, v = egcd(a, self.m)      # g is a non-zero constant since m is irreducible
        return NF(self.m, [x / g[0] for x in u])
    def __truediv__(self, o): return self * self._lift(o).inv()
    def _lift(self, o): return o if isinstance(o, NF) else NF(self.m, [o])
    def __eq__(self, o): return self.c == self._lift(o).c
    def __repr__(self):
        terms = [f'{c}' + ('' if k == 0 else 'a' if k == 1 else f'a^{k}') for k, c in enumerate(self.c) if c]
        return ' + '.join(terms) or '0'

# Q(cbrt 2): m = X^3 - 2
a = NF([-2, 0, 0, 1], [0, 1])
print('1/(1+a)       =', NF(a.m, [1]) / (1 + a))
print('1/(1+a+a^2)   =', NF(a.m, [1]) / (1 + a + a**2), '   (expected a - 1)')
# Q(sqrt2 + sqrt3): m = X^4 - 10 X^2 + 1
al = NF([1, 0, -10, 0, 1], [0, 1])
s2 = (al**3 - 9 * al) * Fraction(1, 2)
print('sqrt2 =', s2, '  and its square:', s2 * s2)
s3 = al - s2
print('sqrt3 =', s3, '  and its square:', s3 * s3)
print('1/(1+sqrt2) =', NF(al.m, [1]) / (1 + s2), '  equals sqrt2 - 1:', NF(al.m, [1]) / (1 + s2) == s2 - 1)

Everything is exact: `s2 * s2` comes out as the constant $2$, although $\sqrt2$ was obtained as a polynomial in $\alpha$ of degree $3$. Inverses cost one extended Euclidean algorithm, exactly as in the proof that $K[X]/(m)$ is a field. This is how SymPy, Sage, PARI or Magma represent algebraic numbers internally (with smarter algorithms for large degrees).

### P8.2

In [ ]:
for e in [sqrt(2) + sqrt(3), sqrt(2) + cbrt(2), 2 * cos(2 * pi / 7), 2 * cos(2 * pi / 9), cos(2 * pi / 17)]:
    print(f'{str(e):18s} {minimal_polynomial(e, X)}')
print()
ok = []
for n in range(3, 31):
    d = sympy.degree(minimal_polynomial(2 * cos(2 * pi / n), X), X)
    ok.append(d == int(totient(n)) // 2)
print('degree of 2cos(2pi/n) equals phi(n)/2 for 3 <= n <= 30:', all(ok))

The polynomial for $\sqrt2+\sqrt[3]2$ is the degree-$6$ polynomial of Exercise 8.1, and $\cos\frac{2\pi}{17}$ has degree $8$, as Exercise 8.4 predicts. The degree of $2\cos\frac{2\pi}n$ is half the degree $\varphi(n)$ of $e^{2i\pi/n}$, for the reason given in the solution of Exercise 8.4: $\zeta$ is a root of $X^2-(\zeta+\zeta^{-1})X+1$.

### P8.3

In [ ]:
def is_pow2(k): return k & (k - 1) == 0
passes = [n for n in range(3, 301) if is_pow2(int(totient(n)))]
fermat = [3, 5, 17, 257, 65537]
gw = set()
for s in range(len(fermat) + 1):
    for ps in itertools.combinations(fermat, s):
        base = math.prod(ps)
        k = 0
        while base * 2**k <= 300:
            if base * 2**k >= 3: gw.add(base * 2**k)
            k += 1
print(passes)
print('same list as 2^k times distinct Fermat primes:', passes == sorted(gw))
print('failing n <= 20:', [n for n in range(3, 21) if n not in gw])

The two lists coincide: $\varphi(n)$ is a power of $2$ exactly when $n=2^kp_1\cdots p_s$ with distinct Fermat primes. (Proof: $\varphi(p^a)=p^{a-1}(p-1)$ is a power of $2$ iff $p=2$, or $a=1$ and $p-1$ is a power of $2$; and $2^j+1$ prime forces $j$ to be a power of $2$, as $2^{ab}+1$ is divisible by $2^a+1$ for odd $b>1$.) Only five Fermat primes are known, and the only $n\le 20$ that fail are $7,9,11,13,14,18,19$. Wantzel's theorem shows that those are not constructible; Chapter 10 shows that all the others are.

### P8.4

In [ ]:
om = sympy.Rational(-1, 2) + sympy.I * sqrt(3) / 2
for c in range(6):
    print(f'c = {c}: degree of cbrt2 + c*omega = {sympy.degree(minimal_polynomial(cbrt(2) + c * om, X), X)}')
print('sqrt2+sqrt3+sqrt5:', sympy.degree(minimal_polynomial(sqrt(2) + sqrt(3) + sqrt(5), X), X))
print('sqrt2+sqrt3+sqrt5+sqrt7:', sympy.degree(minimal_polynomial(sqrt(2) + sqrt(3) + sqrt(5) + sqrt(7), X), X))

Only $c=0$ fails: then $\alpha+c\beta=\sqrt[3]2$ has degree $3$. In the proof of Exercise 8.8, a value of $c$ is excluded when $\alpha_i+c\beta_j=\alpha+c\beta$ for some $j\ne1$; here the roots of $m_\beta=X^2+X+1$ are $\omega,\omega^2$, and $\alpha_i+c\omega^2=\alpha+c\omega$ means $c(\omega^2-\omega)=\alpha-\alpha_i$. For $i=1$ this gives $c=0$; for $i\ne1$, $\alpha-\alpha_i=\sqrt[3]2(1-\omega^{\pm1})$ is not a rational multiple of $\omega^2-\omega=-i\sqrt3$, so no other rational $c$ is excluded. The sums of square roots of distinct primes have the full degree $2^k$, a theorem of Besicovitch (1940).

### P8.5

In [ ]:
def line_line(p1, p2, q1, q2):
    d1, d2 = p2 - p1, q2 - q1
    den = (d1.conjugate() * d2).imag
    if abs(den) < 1e-12: return []
    return [p1 + ((q1 - p1).conjugate() * d2).imag / den * d1]

def line_circle(p1, p2, c, r):
    d, f = p2 - p1, p1 - c
    a, b, cc = abs(d)**2, 2 * (f.conjugate() * d).real, abs(f)**2 - r * r
    disc = b * b - 4 * a * cc
    if disc < -1e-12: return []
    s = 0.0 if disc < 1e-10 else math.sqrt(disc)       # tangent: snap to a double root
    return [p1 + (-b + s) / (2 * a) * d, p1 + (-b - s) / (2 * a) * d]

def circle_circle(c1, r1, c2, r2):
    d = abs(c2 - c1)
    if d < 1e-12 or d > r1 + r2 + 1e-12 or d < abs(r1 - r2) - 1e-12: return []
    a = (r1 * r1 - r2 * r2 + d * d) / (2 * d); h2 = r1 * r1 - a * a; h = 0.0 if h2 < 1e-10 else math.sqrt(h2)
    u = (c2 - c1) / d; m = c1 + a * u
    return [m + 1j * h * u, m - 1j * h * u]

class PointSet:
    # points up to a tolerance, with a grid of buckets for fast lookups
    def __init__(self, tol=1e-6): self.tol, self.pts, self.grid = tol, [], {}
    def add(self, z):
        i, j = int(math.floor(z.real / self.tol)), int(math.floor(z.imag / self.tol))
        for di in (-1, 0, 1):
            for dj in (-1, 0, 1):
                for w in self.grid.get((i + di, j + dj), []):
                    if abs(w - z) < self.tol: return
        self.grid.setdefault((i, j), []).append(z); self.pts.append(z)

def one_round(pts):
    lines = {}
    for p, q in itertools.combinations(pts, 2):
        d = (q - p) / abs(q - p)
        if d.real < -1e-12 or (abs(d.real) < 1e-12 and d.imag < 0): d = -d
        lines[(round(d.real, 7), round(d.imag, 7), round((p * d.conjugate()).imag, 7))] = (p, q)
    circles = {}
    for c in pts:
        for q in pts:
            if q != c: circles[(round(c.real, 7), round(c.imag, 7), round(abs(q - c), 7))] = (c, abs(q - c))
    L, C = list(lines.values()), list(circles.values())
    S = PointSet()
    for z in pts: S.add(z)
    for (a, b), (c, d) in itertools.combinations(L, 2):
        for z in line_line(a, b, c, d): S.add(z)
    for a, b in L:
        for c, r in C:
            for z in line_circle(a, b, c, r): S.add(z)
    for (c1, r1), (c2, r2) in itertools.combinations(C, 2):
        for z in circle_circle(c1, r1, c2, r2): S.add(z)
    return S.pts, len(L), len(C)

P = [0j, 1 + 0j]
history = [P]
for k in range(2):
    P, nl, nc = one_round(P)
    history.append(P)
    print(f'round {k + 1}: {nl} lines, {nc} circles, {len(P)} points')

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter([z.real for z in history[2]], [z.imag for z in history[2]], s=6, label='after 2 rounds')
ax.scatter([z.real for z in history[1]], [z.imag for z in history[1]], s=30, c='C1', label='after 1 round')
ax.set_aspect('equal'); ax.legend(loc='upper left'); ax.set_xlim(-3.5, 4.5); ax.set_ylim(-2.5, 2.5)
plt.show()

One round gives $6$ points ($0,1,-1,2$ and $\frac12\pm i\frac{\sqrt3}2$), two rounds give $203$. A third round gives $1\,723\,816\,861$ points (sequence A333944 in the OEIS, computed in 2020 with exact arithmetic, as floating point is no longer trustworthy at that scale). Every one of these numbers lies in a tower of quadratic extensions of $\Q$ (Theorem 8.17), and none of them is $\sqrt[3]2$, at any round.

Near-tangencies are the numerical danger here: when a line touches a circle, the discriminant is $0$ up to rounding errors of order $10^{-16}$, and its square root is of order $10^{-8}$, so a tangency point appears as two points $10^{-8}$ apart unless we snap tiny discriminants to $0$. Without the snapping, the count after two rounds comes out as $211$ instead of $203$.

Now the pentagon, with the steps of Figure 8.1 (unit circle; the circle centred at $-\frac12$ through $i$; the perpendicular bisector of $[0,t]$, obtained from two circles):

In [ ]:
O, one = 0j, 1 + 0j
minus1 = [z for z in line_circle(O, one, O, 1) if z.real < 0][0]
i_ = [z for z in circle_circle(minus1, 2, one, 2) if z.imag > 0]           # perpendicular bisector of [-1, 1] ...
i_ = [z for z in line_circle(O, i_[0], O, 1) if z.imag > 0][0]             # ... meets the unit circle at i
half = line_line(O, one, *circle_circle(O, 1, minus1, 1))[0]               # -1/2, bisector of [-1, 0]
t = [z for z in line_circle(O, one, half, abs(i_ - half)) if z.real > 0][0]
print('t =', t.real, ' (sqrt5-1)/2 =', (math.sqrt(5) - 1) / 2)
A, B = circle_circle(O, abs(t), t, abs(t))                                  # bisector of [0, t]
v = [z for z in line_circle(A, B, O, 1)]
print('vertices found:', [complex(round(z.real, 12), round(z.imag, 12)) for z in v])
print('expected      :', [complex(round(math.cos(a), 12), round(math.sin(a), 12)) for a in (2 * math.pi / 5, -2 * math.pi / 5)])

The line $x=\cos72^\circ$ meets the unit circle at $e^{\pm2i\pi/5}$; the other vertices follow by stepping around the circle with circles of radius $|e^{2i\pi/5}-1|$. Every step used only lines through known points and circles centred at known points through known points, which is exactly the setting of Proposition 8.16 and Theorem 8.17.